In [1]:
import pandas as pd


In [2]:
df = pd.read_csv("sales.csv")
df


,order_id,order_date,customer_name,product_name,category,region,quantity,sales,profit,discount
0,1,2024-01-05,Amit Sharma,Office Chair,Furniture,West,1,3500,1000,0.10
1,2,2024-01-10,Neha Verma,Printer,Technology,East,1,18000,3000,0.05
2,3,2024-01-15,Rahul Singh,Notebook,Stationery,South,10,500,200,0.00
3,4,2024-02-05,Amit Sharma,Table,Furniture,West,1,8000,2000,0.15
4,5,2024-02-10,Neha Verma,Laptop,Technology,East,1,55000,8000,0.10
5,6,2024-02-15,Rahul Singh,Pen,Stationery,South,20,600,150,0.05
6,7,2024-03-01,Amit Sharma,Monitor,Technology,North,1,12000,2500,0.08
7,8,2024-03-05,Neha Verma,Office Chair,Furniture,East,2,7000,1800,0.12
8,9,2024-03-10,Rahul Singh,Notebook,Stationery,South,15,750,300,0.00
9,10,2024-03-15,Amit Sharma,Desk,Furniture,West,1,15000,3500,0.10


In [5]:
# Handle missing values and remove duplicates
df.isnull().sum()
df.drop_duplicates(inplace=True)
df.fillna(0, inplace=True)



In [11]:
# Create Profit column as Sales minus Cost
df.columns




Index(['order_id', 'order_date', 'customer_name', 'product_name', 'category',
       'region', 'quantity', 'sales', 'profit', 'discount', 'cost'],
      dtype='object')

In [14]:
# Generate quarter-wise revenue trend
df["order_date"] = pd.to_datetime(df["order_date"])
df["quarter"] = df["order_date"].dt.to_period("Q")

quarterly_revenue = df.groupby("quarter")["sales"].sum()
quarterly_revenue




quarter
2024Q1    120350
Freq: Q-DEC, Name: sales, dtype: int64

In [15]:
# Detect Sales Outliers (IQR)
Q1 = df["sales"].quantile(0.25)
Q3 = df["sales"].quantile(0.75)
IQR = Q3 - Q1

outliers = df[
    (df["sales"] < Q1 - 1.5 * IQR) |
    (df["sales"] > Q3 + 1.5 * IQR)
]

outliers


,order_id,order_date,customer_name,product_name,category,region,quantity,sales,profit,discount,cost,quarter
4,5,2024-02-10,Neha Verma,Laptop,Technology,East,1,55000,16500.0,0.1,38500.0,2024Q1


In [18]:
# Forecast next 3 months using moving average
monthly_sales = df.resample("ME", on="order_date")["sales"].sum()
monthly_sales
forecast_next_3 = monthly_sales.rolling(3).mean().tail(3)
forecast_next_3



order_date
2024-01-31             NaN
2024-02-29             NaN
2024-03-31    40116.666667
Freq: ME, Name: sales, dtype: float64

In [20]:
# Segment customers into low, medium, and high spenders
customer_sales = df.groupby("customer_name")["sales"].sum()
customer_sales
bins = [0, 5000, 20000, customer_sales.max()]
labels = ["Low", "Medium", "High"]

customer_segments = pd.cut(customer_sales, bins=bins, labels=labels)
customer_segments



customer_name
Amit Sharma    High
Neha Verma     High
Rahul Singh     Low
Name: sales, dtype: category
Categories (3, object): ['Low' < 'Medium' < 'High']